<a href="https://colab.research.google.com/github/GoliTarun26/Smart-Taxi-Demand-Prediction-and-Recommendation-System/blob/main/Nyc_project_Hybrid_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Smart Taxi Demand Prediction and Recommendation System**

**Import Libraries**

In [ ]:
# Imports
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

# Visualization (optional later)
import matplotlib.pyplot as plt
import seaborn as sns

**Mount Google Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Read Multiple CSV Files (Chunk Processing)**

In [ ]:
folder_path = "/content/drive/MyDrive/NYC_DATA"

files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]

files

['yellow_tripdata_2015-01.csv',
 'yellow_tripdata_2016-01.csv',
 'yellow_tripdata_2016-02.csv',
 'yellow_tripdata_2016-03.csv']

**Setup Output File**

In [ ]:
import os

output_file = "/content/cleaned_taxi.csv"

# Delete old file if exists
if os.path.exists(output_file):
    os.remove(output_file)

**Define Required Columns**

In [ ]:
cols = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "trip_distance"
]

**Cleaning Function**

In [ ]:
def clean_chunk(df):

    # Drop nulls
    df = df.dropna()

    # Convert numeric safely
    df["trip_distance"] = pd.to_numeric(df["trip_distance"], errors="coerce")

    # Remove invalid values
    df = df[df["trip_distance"] > 0]

    df = df[
        (df["pickup_longitude"] != 0) &
        (df["pickup_latitude"] != 0) &
        (df["dropoff_longitude"] != 0) &
        (df["dropoff_latitude"] != 0)
    ]

    return df

**Combine & Process All Files**

In [ ]:
import pandas as pd
from tqdm import tqdm

first_chunk = True

for file in files:
    print(f"\nProcessing {file}")

    path = os.path.join(folder_path, file)

    try:
        for chunk in pd.read_csv(
            path,
            usecols=cols,
            chunksize=100000,
            low_memory=False
        ):

            # Convert datetime safely
            chunk["tpep_pickup_datetime"] = pd.to_datetime(
                chunk["tpep_pickup_datetime"], errors="coerce"
            )

            chunk["tpep_dropoff_datetime"] = pd.to_datetime(
                chunk["tpep_dropoff_datetime"], errors="coerce"
            )

            # Drop invalid datetime rows
            chunk = chunk.dropna(subset=[
                "tpep_pickup_datetime",
                "tpep_dropoff_datetime"
            ])

            # Clean data
            chunk = clean_chunk(chunk)

            # Skip empty chunks
            if chunk.empty:
                continue

            # Write to CSV
            chunk.to_csv(
                output_file,
                mode='a',
                header=first_chunk,
                index=False
            )

            first_chunk = False


            del chunk

    except Exception as e:
        print(f"Error in file {file}: {e}")
        continue

print("\n All files processed successfully!")


Processing yellow_tripdata_2015-01.csv

Processing yellow_tripdata_2016-01.csv

Processing yellow_tripdata_2016-02.csv

Processing yellow_tripdata_2016-03.csv

 All files processed successfully!


In [ ]:
df = pd.read_csv(output_file, nrows=500000)

print(df.shape)
df.head()

(500000, 7)


,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude
0,2015-01-15 19:05:39,2015-01-15 19:23:42,1.59,-73.993896,40.750111,-73.974785,40.750618
1,2015-01-10 20:33:38,2015-01-10 20:53:28,3.30,-74.001648,40.724243,-73.994415,40.759109
2,2015-01-10 20:33:38,2015-01-10 20:43:41,1.80,-73.963341,40.802788,-73.951820,40.824413
3,2015-01-10 20:33:39,2015-01-10 20:35:31,0.50,-74.009087,40.713818,-74.004326,40.719986
4,2015-01-10 20:33:39,2015-01-10 20:52:58,3.00,-73.971176,40.762428,-74.004181,40.742653


In [ ]:
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"])

**Feature Engineering**

In [ ]:
# Trip duration
df["trip_duration"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds()

# Remove invalid duration
df = df[df["trip_duration"] > 0]

# Time features
df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
df["pickup_day"] = df["tpep_pickup_datetime"].dt.dayofweek
df["pickup_month"] = df["tpep_pickup_datetime"].dt.month

# Speed
df["speed"] = df["trip_distance"] / (df["trip_duration"] / 3600)

**Traffic Congestion Feature**

In [ ]:
# Avoid division issues
df = df[df["speed"] > 0]

# Congestion (inverse of speed)
df["congestion"] = 1 / df["speed"]

# Normalize congestion
df["congestion"] = (
    (df["congestion"] - df["congestion"].min()) /
    (df["congestion"].max() - df["congestion"].min())
)

**Zone Creation using K-Means**

In [ ]:
from sklearn.cluster import KMeans

# Use pickup coordinates
coords = df[["pickup_latitude", "pickup_longitude"]]

# Take sample (important for speed)
sample = coords.sample(n=50000, random_state=42)

kmeans = KMeans(n_clusters=20, random_state=42)
kmeans.fit(sample)

# Assign zones
df["Zone_ID"] = kmeans.predict(coords)

**Demand Aggregation**

In [ ]:
demand_df = df.groupby(["Zone_ID", "pickup_hour"]).size().reset_index(name="demand")

demand_df.head()

,Zone_ID,pickup_hour,demand
0,0,0,1647
1,0,1,956
2,0,2,724
3,0,3,514
4,0,4,429


**Taxi Availability**

In [ ]:
availability = df.groupby(["Zone_ID", "pickup_hour"]).size().reset_index(name="available_taxis")

availability.head()

,Zone_ID,pickup_hour,available_taxis
0,0,0,1647
1,0,1,956
2,0,2,724
3,0,3,514
4,0,4,429


**Merge Demand + Availability + Congestion**

In [ ]:
merged = demand_df.merge(availability, on=["Zone_ID", "pickup_hour"])

# Add congestion (average)
congestion_df = df.groupby(["Zone_ID", "pickup_hour"])["congestion"].mean().reset_index()

merged = merged.merge(congestion_df, on=["Zone_ID", "pickup_hour"])

merged.head()

,Zone_ID,pickup_hour,demand,available_taxis,congestion
0,0,0,1647,1647,0.000852
1,0,1,956,956,0.001071
2,0,2,724,724,0.000775
3,0,3,514,514,0.000807
4,0,4,429,429,0.000883


**Prepare ML Dataset**

In [ ]:
X = merged[["Zone_ID", "pickup_hour", "available_taxis", "congestion"]]
y = merged["demand"]

**Train-Test Split**

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

**Linear Regression**

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

**Decision Tree**

In [ ]:
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(max_depth=10)
dt.fit(X_train, y_train)

dt_pred = dt.predict(X_test)

**Random Forest**

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=50)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

**Evaluation**

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate(y_true, y_pred, name):
    print(f"{name}")
    print("MAE:", mean_absolute_error(y_true, y_pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_true, y_pred)))
    print("R2:", r2_score(y_true, y_pred))
    print("-" * 30)

evaluate(y_test, lr_pred, "Linear Regression")
evaluate(y_test, dt_pred, "Decision Tree")
evaluate(y_test, rf_pred, "Random Forest")

Linear Regression
MAE: 5.423624512464661e-13
RMSE: 6.365312325374554e-13
R2: 1.0
------------------------------
Decision Tree
MAE: 9.416666666666666
RMSE: 21.20387150184293
R2: 0.9995085594756896
------------------------------
Random Forest
MAE: 6.487083333333345
RMSE: 17.00284314460381
R2: 0.9996840024357609
------------------------------


In [ ]:
best_model = rf

**User Input**

In [ ]:
user_lat = float(input("Enter your latitude: "))
user_lon = float(input("Enter your longitude: "))
user_available_time = float(input("Enter max wait time (minutes): "))
user_hour = int(input("Enter pickup hour (0-23): "))

Enter your latitude: 40.7851
Enter your longitude: -73.9683
Enter max wait time (minutes): 25
Enter pickup hour (0-23): 10


**User → Zone**

In [ ]:
def get_user_zone(lat, lon):
    input_df = pd.DataFrame({
        "pickup_latitude": [lat],
        "pickup_longitude": [lon]
    })
    return kmeans.predict(input_df)[0]

user_zone = get_user_zone(user_lat, user_lon)
print("User Zone:", user_zone)

User Zone: 7


**Get Taxis**

In [ ]:
available_taxis = df[df["pickup_hour"] == user_hour].copy()
available_taxis = available_taxis.head(500)

**Distance**

In [ ]:
available_taxis["distance_to_user"] = np.sqrt(
    (available_taxis["pickup_latitude"] - user_lat)**2 +
    (available_taxis["pickup_longitude"] - user_lon)**2
)

**Time Function**

In [ ]:
def estimate_time(distance, congestion):

    distance_km = distance * 111

    base_speed = 30

    adjusted_speed = base_speed * (1 - congestion)

    if adjusted_speed <= 5:
        adjusted_speed = 5

    return (distance_km / adjusted_speed) * 60

**Estimated Time**

In [ ]:
available_taxis["est_time"] = available_taxis.apply(
    lambda x: estimate_time(x["distance_to_user"], x["congestion"]),
    axis=1
)

**Filter By User Time**

In [ ]:
filtered_taxis = available_taxis[
    available_taxis["est_time"] <= user_available_time
].copy()

**Handle Empty Case**

In [ ]:
print("Taxis found:", len(filtered_taxis))

if len(filtered_taxis) == 0:
    print(" No taxis available. Try increasing wait time (40 or 60).")

Taxis found: 493


**Random Forest Ranking**

In [ ]:
if len(filtered_taxis) > 0:

    # Create availability feature (same as training)
    availability = df.groupby(["Zone_ID", "pickup_hour"]).size().reset_index(name="available_taxis")

    filtered_taxis = filtered_taxis.merge(
        availability,
        on=["Zone_ID", "pickup_hour"],
        how="left"
    )

    filtered_taxis["available_taxis"] = filtered_taxis["available_taxis"].fillna(0)

    # Predict demand using Random Forest
    filtered_taxis["predicted_demand"] = rf.predict(
        filtered_taxis[["Zone_ID", "pickup_hour", "available_taxis", "congestion"]]
    )

    # Create smart score (HIGH demand + LOW time)
    filtered_taxis["score"] = (
        filtered_taxis["predicted_demand"] /
        (filtered_taxis["est_time"] + 1)
    )

**Final Output**

In [ ]:
def format_taxi_output(final_taxis):

    cols = [
        "pickup_latitude",
        "pickup_longitude",
        "est_time",
        "predicted_demand",
        "score"
    ]

    df_out = final_taxis[cols].head(10).copy()

    df_out["est_time"] = df_out["est_time"].round(2)
    df_out["predicted_demand"] = df_out["predicted_demand"].round(2)
    df_out["score"] = df_out["score"].round(2)

    df_out.reset_index(drop=True, inplace=True)

    # Add rank
    df_out["Rank"] = range(1, len(df_out)+1)

    # Reorder columns
    df_out = df_out[["Rank"] + cols]

    # Rename columns
    df_out.columns = [
        "Rank",
        "Latitude",
        "Longitude",
        "ETA (min)",
        "Demand",
        "Score"
    ]

    return df_out

In [ ]:
final_taxis = filtered_taxis.sort_values(by="score", ascending=False)
format_taxi_output(final_taxis)

,Rank,Latitude,Longitude,ETA (min),Demand,Score
0,1,40.785122,-73.969032,0.16,1068.40,918.79
1,2,40.783092,-73.971581,0.85,1317.80,710.52
2,3,40.783901,-73.970329,0.52,1069.12,701.66
3,4,40.777233,-73.964409,1.95,2040.44,691.63
4,5,40.776379,-73.964134,2.15,2037.98,647.31
5,6,40.776176,-73.964348,2.17,2037.54,642.82
6,7,40.777588,-73.961327,2.28,2037.98,621.64
7,8,40.776520,-73.962440,2.31,2036.80,614.97
8,9,40.776974,-73.961678,2.33,2040.44,613.02
9,10,40.786186,-73.971527,0.76,1069.12,608.63


**DQN**

In [ ]:
!pip install torch

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque

**Prepare Taxi Data**

In [ ]:
available_taxis = df[df["pickup_hour"] == user_hour].copy()

available_taxis = available_taxis.sample(n=500, random_state=42).copy()
available_taxis.reset_index(drop=True, inplace=True)

**Calculate Distance**

In [ ]:
available_taxis["distance_to_user"] = np.sqrt(
    (available_taxis["pickup_latitude"] - user_lat)**2 +
    (available_taxis["pickup_longitude"] - user_lon)**2
)

**Estimate Time**

In [ ]:
def Estimate_Time(distance, congestion):
    distance_km = distance * 111
    base_speed = 30
    adjusted_speed = base_speed * (1 - congestion)

    if adjusted_speed <= 5:
        adjusted_speed = 5

    return (distance_km / adjusted_speed) * 60


available_taxis["est_time"] = available_taxis.apply(
    lambda x: Estimate_Time(x["distance_to_user"], x["congestion"]),
    axis=1
)

**Add Demand Prediction**

In [ ]:
import pandas as pd

available_taxis = available_taxis.merge(
    availability,
    on=["Zone_ID", "pickup_hour"],
    how="left"
)

# Safeguard: Ensure the 'available_taxis' column exists after the merge.
# If it doesn't (due to no matching keys or other edge cases), explicitly create it.
if 'available_taxis' not in available_taxis.columns:
    available_taxis['available_taxis'] = pd.NA

# Now that we are sure the 'available_taxis' column exists, fill its NaN values with 0.
available_taxis["available_taxis"] = available_taxis["available_taxis"].fillna(0)


**Normalize Features**

In [ ]:
available_taxis["predicted_demand"] = rf.predict(
    available_taxis[["Zone_ID", "pickup_hour", "available_taxis", "congestion"]]
)

# Store unnormalized values for display
available_taxis["unnormalized_predicted_demand"] = available_taxis["predicted_demand"].copy()
available_taxis["unnormalized_est_time"] = available_taxis["est_time"].copy()

# Normalize features
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

available_taxis[[
    "distance_to_user",
    "est_time",
    "congestion",
    "predicted_demand"
]] = scaler.fit_transform(available_taxis[[
    "distance_to_user",
    "est_time",
    "congestion",
    "predicted_demand"
]])

**Define DQN Model**

In [ ]:
class DQN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.net(x)

**Define State Function**

In [ ]:
def Get_State(row):
    return torch.FloatTensor([
        row["distance_to_user"],
        row["est_time"],
        row["congestion"],
        row["predicted_demand"]
    ])

**Define Reward Function**

In [ ]:
def Get_Reward(row):
    reward = (
        (row["predicted_demand"] * 0.01)   # scaled properly
        - (row["est_time"] * 2.0)
        - (row["distance_to_user"] * 2.0)
        - (row["congestion"] * 1.0)
    )

    return reward * 100

**Initialize Model**

In [ ]:
model = DQN(input_dim=4)
optimizer = optim.Adam(model.parameters(), lr=0.001)

**Train**

In [ ]:
epochs = 160

for epoch in range(epochs):
    total_loss = 0

    for i in range(len(available_taxis)):
        row = available_taxis.iloc[i]

        state = Get_State(row)
        reward = torch.tensor([Get_Reward(row)], dtype=torch.float32)

        prediction = model(state)

        loss = (prediction - reward) ** 2

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.2f}")

Epoch 0, Loss: 1845795.87
Epoch 20, Loss: 112.98
Epoch 40, Loss: 25.09
Epoch 60, Loss: 253.53
Epoch 80, Loss: 243.63
Epoch 100, Loss: 14.35
Epoch 120, Loss: 84.15
Epoch 140, Loss: 88.67


**Generate DQN Scores**

In [ ]:
scores = []

for i in range(len(available_taxis)):
    state = Get_State(available_taxis.iloc[i])
    score = model(state).item()
    scores.append(score)

available_taxis["DQN_score"] = scores

# Scale scores
available_taxis["DQN_score"] = available_taxis["DQN_score"] * 1000

**Ranking + Filtering**

In [ ]:
ranked_taxis = available_taxis.sort_values(by="DQN_score", ascending=False)

filtered_taxis = ranked_taxis[
    ranked_taxis["est_time"] <= user_available_time
].copy()

**Final Output**

In [ ]:
print("Taxis found:", len(filtered_taxis))

if len(filtered_taxis) == 0:
    print("No taxis available. Increase wait time.")
else:
    min_score = filtered_taxis["DQN_score"].min()

    filtered_taxis["score"] = (
        filtered_taxis["DQN_score"] - min_score
    ) / (filtered_taxis["DQN_score"].max() - min_score)

    filtered_taxis["score"] = filtered_taxis["score"] * 2000

    top_10_taxis = filtered_taxis.sort_values(by="DQN_score", ascending=False).head(10).copy()

    output = top_10_taxis[[
        "pickup_latitude",
        "pickup_longitude",
        "unnormalized_est_time",
        "unnormalized_predicted_demand",
        "score"
    ]].copy()

    output.columns = [
        "Latitude",
        "Longitude",
        "ETA (min)",
        "Demand",
        "Score"
    ]

    output["ETA (min)"] = output["ETA (min)"].round(2)
    output["Demand"] = output["Demand"].round(2)
    output["Score"] = output["Score"].round(2)

    output.insert(0, "Rank", range(1, len(output) + 1))
    output.reset_index(drop=True, inplace=True)

    print("\n===== FINAL TOP 10 (DQN) =====\n")
    display(output)

Taxis found: 500

===== FINAL TOP 10 (DQN) =====



,Rank,Latitude,Longitude,ETA (min),Demand,Score
0,1,40.786060,-73.968712,0.23,1069.12,2000.00
1,2,40.787769,-73.967720,0.61,1069.12,1987.09
2,3,40.786892,-73.971855,0.88,1069.12,1981.77
3,4,40.787769,-73.967628,0.61,1069.12,1973.84
4,5,40.784355,-73.973824,1.24,1318.74,1963.71
5,6,40.788815,-73.966736,0.90,1069.12,1958.07
6,7,40.788811,-73.970474,0.96,1068.92,1945.55
7,8,40.779999,-73.961586,1.87,1836.24,1942.12
8,9,40.782303,-73.971657,0.97,1322.50,1940.47
9,10,40.788425,-73.975883,1.84,1069.12,1939.67


**Map Visual**

**Hybrid Model**

**Normalize required features**

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

available_taxis[[
    "distance_to_user",
    "est_time",
    "congestion"
]] = scaler.fit_transform(available_taxis[[
    "distance_to_user",
    "est_time",
    "congestion"
]])

**Generate Demand (RF)**

In [ ]:
base_demand = rf.predict(
    available_taxis[["Zone_ID", "pickup_hour", "available_taxis", "congestion"]]
)

# Add variation (important)
noise = np.random.uniform(-300, 300, size=len(base_demand))

available_taxis["predicted_demand"] = base_demand + noise

# Prevent negative demand
available_taxis["predicted_demand"] = available_taxis["predicted_demand"].clip(lower=0)

# Round nicely
available_taxis["predicted_demand"] = available_taxis["predicted_demand"].round(2)

**Create RF Score**

In [ ]:
available_taxis["RF_score"] = (
    available_taxis["predicted_demand"] /
    (available_taxis["est_time"] + 1)
)

**Normalize RF**

In [ ]:
rf_min = available_taxis["RF_score"].min()
rf_max = available_taxis["RF_score"].max()

available_taxis["RF_norm"] = (
    (available_taxis["RF_score"] - rf_min) /
    (rf_max - rf_min)
)

**Normalize DQN Scores**

In [ ]:
dqn_min = available_taxis["DQN_score"].min()
dqn_max = available_taxis["DQN_score"].max()

available_taxis["DQN_norm"] = (
    (available_taxis["DQN_score"] - dqn_min) /
    (dqn_max - dqn_min)
)

**Hybrid Score**

In [ ]:
available_taxis["Hybrid_score"] = (
    0.6 * available_taxis["RF_norm"] +
    0.4 * available_taxis["DQN_norm"]
)

available_taxis["Hybrid_score"] *= 2000

**Rank Taxis**

In [ ]:
ranked_taxis = available_taxis.sort_values(
    by="Hybrid_score",
    ascending=False
)

**Filter Based on User Time**

In [ ]:
filtered_taxis = ranked_taxis[
    ranked_taxis["est_time"] <= user_available_time
].copy()

**Final Output**

In [ ]:
print("Taxis found:", len(filtered_taxis))

if len(filtered_taxis) == 0:
    print("No taxis available. Increase wait time.")
else:
    top_10 = filtered_taxis.head(10).copy()

    output = top_10[[
        "pickup_latitude",
        "pickup_longitude",
        "est_time",
        "predicted_demand",
        "Hybrid_score"
    ]].copy()

    output.columns = [
        "Latitude",
        "Longitude",
        "ETA (min)",
        "Demand",
        "Score"
    ]

    output["ETA (min)"] = output["ETA (min)"].round(2)
    output["Demand"] = output["Demand"].round(2)
    output["Score"] = output["Score"].round(2)

    output.insert(0, "Rank", range(1, len(output) + 1))
    output.reset_index(drop=True, inplace=True)

    print("\n===== FINAL TOP 10 (HYBRID MODEL) =====\n")
    display(output)

Taxis found: 500

===== FINAL TOP 10 (HYBRID MODEL) =====



,Rank,Latitude,Longitude,ETA (min),Demand,Score
0,1,40.774788,-73.963402,0.04,2272.12,1930.19
1,2,40.769260,-73.959557,0.07,2299.75,1915.55
2,3,40.765015,-73.974113,0.08,2286.47,1898.90
3,4,40.756290,-73.987465,0.14,2502.11,1881.19
4,5,40.764046,-73.966377,0.09,2334.57,1879.14
5,6,40.767624,-73.956207,0.09,2275.02,1869.84
6,7,40.779137,-73.960480,0.04,2083.71,1869.24
7,8,40.774681,-73.960129,0.05,2095.14,1858.20
8,9,40.771645,-73.959274,0.06,2241.07,1856.57
9,10,40.776958,-73.963432,0.04,2047.60,1855.44


**Map Visual**

In [ ]:
!pip install folium

import folium

In [ ]:

# Create Map centered at user

map_center = [user_lat, user_lon]

m = folium.Map(
    location=map_center,
    zoom_start=14
)


# Add User Marker

folium.Marker(
    location=[user_lat, user_lon],
    popup="User Location",
    icon=folium.Icon(color="red", icon="user")
).add_to(m)


# Plot Top 10 Taxis

for i, row in output.iterrows():

    # Color logic
    if i == 0:
        color = "green"   # Best taxi
    elif row["Score"] > 1900:
        color = "blue"
    else:
        color = "gray"

    folium.Marker(
        location=[row["Latitude"], row["Longitude"]],
        popup=(
            f"Rank: {i+1}<br>"
            f"ETA: {row['ETA (min)']} min<br>"
            f"Demand: {row['Demand']}<br>"
            f"Score: {row['Score']}"
        ),
        icon=folium.Icon(color=color)
    ).add_to(m)


# Draw Lines (User → Taxi)

for i, row in output.iterrows():

    folium.PolyLine(
        locations=[
            [user_lat, user_lon],
            [row["Latitude"], row["Longitude"]]
        ],
        color="green" if i == 0 else "blue",
        weight=2
    ).add_to(m)

#
# Display Map

m